# 04 最终候选生成、特征工程与模型评估

本 Notebook 是唯一建模入口，直接交付五路召回、31 特征、LightGBM LambdaRank 最终模型，并与两组公开 baseline 在同一测试期和用户切分下比较。

In [1]:

import json
import os
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-hm-recsys')
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 160
plt.rcParams["savefig.bbox"] = "tight"
np.random.seed(2026)

PROJECT_DIR = next((base / 'optimization' for base in (Path.cwd(), *Path.cwd().parents) if (base / 'optimization' / 'code').is_dir()), Path.cwd())
CLEAN_DIR = PROJECT_DIR / "output" / "clean"
MINING_DIR = PROJECT_DIR / "output" / "mining"
FINAL_DIR = PROJECT_DIR / "output" / "modeling" / "final_model"
TABLE_DIR = FINAL_DIR / "tables"
FIGURE_DIR = FINAL_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

transactions = pd.read_parquet(CLEAN_DIR / "transactions_clean.parquet")
customers = pd.read_parquet(CLEAN_DIR / "customers_clean.parquet")
articles = pd.read_parquet(CLEAN_DIR / "articles_clean.parquet")
transactions["article_num"] = transactions["article_id"].astype("int32")
transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])
articles["article_num"] = articles["article_id"].astype("int32")

TARGET_END = transactions["t_dat"].max()
TARGET_START = TARGET_END - pd.Timedelta(days=6)
history = transactions[transactions["t_dat"] < TARGET_START].copy()
target = transactions[transactions["t_dat"] >= TARGET_START].copy()
target_users = np.sort(target["customer_key"].unique())
target_user_set = set(target_users)
target_truth = target[["customer_key", "article_num"]].drop_duplicates()
target_truth["label"] = np.int8(1)
history_pair_set = history[["customer_key", "article_num"]].drop_duplicates()
history_pair_set["in_history"] = np.int8(1)


def top_k_by_user(frame, score_column, top_n=12, article_column="article_num", ascending=False):
    ordered = frame.sort_values(
        ["customer_key", score_column, article_column], ascending=[True, ascending, True], kind="stable"
    )
    ordered["_rank"] = ordered.groupby("customer_key", sort=False, observed=True).cumcount() + 1
    return ordered[ordered["_rank"] <= top_n].drop(columns="_rank")


def evaluate_candidates(name, candidates, users=None, k=12, return_frame=False):
    ranked = candidates[["customer_key", "article_num", "score"]].copy()
    target_truth_for_eval = target_truth
    if users is not None:
        allowed_users = set(np.asarray(users, dtype=np.uint64).tolist())
        ranked = ranked[ranked["customer_key"].isin(allowed_users)].copy()
        target_truth_for_eval = target_truth[target_truth["customer_key"].isin(allowed_users)].copy()
    ranked = ranked.sort_values(
        ["customer_key", "score", "article_num"], ascending=[True, False, True], kind="stable"
    )
    ranked = ranked.drop_duplicates(["customer_key", "article_num"], keep="first")
    ranked["_rank"] = ranked.groupby("customer_key", sort=False, observed=True).cumcount() + 1
    ranked = ranked[ranked["_rank"] <= k]
    evaluated = ranked.merge(target_truth, on=["customer_key", "article_num"], how="left")
    evaluated["label"] = evaluated["label"].fillna(0).astype("int8")
    evaluated["cumulative_hits"] = evaluated.groupby("customer_key", sort=False)["label"].cumsum()
    evaluated["precision_at_hit"] = evaluated["label"] * evaluated["cumulative_hits"] / evaluated["_rank"]
    user_hits = evaluated.groupby("customer_key", observed=True).agg(
        hit_sum=("label", "sum"), ap_sum=("precision_at_hit", "sum")
    ).reset_index()
    truth_counts = target_truth_for_eval.groupby("customer_key", observed=True)["label"].sum().rename("truth_count").reset_index()
    user_hits = user_hits.merge(truth_counts, on="customer_key", how="right").fillna({"hit_sum": 0, "ap_sum": 0})
    user_hits["recall"] = user_hits["hit_sum"] / user_hits["truth_count"].clip(lower=1)
    user_hits["average_precision"] = user_hits["ap_sum"] / user_hits["truth_count"].clip(lower=1, upper=k)
    result = {
        "channel": name,
        f"map@{k}": float(user_hits["average_precision"].mean()),
        f"recall@{k}": float(user_hits["recall"].mean()),
        f"hit_rate@{k}": float((user_hits["hit_sum"] > 0).mean()),
        "covered_users": int(ranked["customer_key"].nunique()),
        "evaluated_users": int(len(user_hits)),
    }
    if return_frame:
        return result, ranked
    print(json.dumps(result, ensure_ascii=False))
    return result


print("transactions:", transactions.shape)
print("history:", history.shape, history["t_dat"].min().date(), history["t_dat"].max().date())
print("target:", target.shape, TARGET_START.date(), TARGET_END.date())
print("target users:", len(target_users), "target pairs:", len(target_truth))
print("LightGBM:", lgb.__version__)


transactions: (31788324, 6)
history: (31548013, 6) 2018-09-20 2020-09-15
target: (240311, 6) 2020-09-16 2020-09-22
target users: 68984 target pairs: 213728
LightGBM: 4.7.0


## 1. 公开 Baseline 本地复现

In [2]:
MODEL_DIR = FINAL_DIR
baseline_started = time.time()
target_user_frame = pd.DataFrame({'customer_key': target_users})
user_fold = target_user_frame.copy()
user_fold['fold'] = np.where(user_fold['customer_key'].astype(np.uint64) % 10 < 8, 'train', 'valid')
train_users = user_fold.loc[user_fold['fold'] == 'train', 'customer_key'].to_numpy(dtype=np.uint64)
valid_users = user_fold.loc[user_fold['fold'] == 'valid', 'customer_key'].to_numpy(dtype=np.uint64)

# Baseline 1: 1/2/3-week repeat purchase + window popularity fallback.
assigned_users = set()
baseline1_chunks = []
for window_days in [7, 14, 21]:
    window = history[history['t_dat'] >= TARGET_START - pd.Timedelta(days=window_days)]
    counts = window.groupby(['customer_key', 'article_num'], as_index=False, observed=True).agg(
        purchase_count=('t_dat', 'size'), last_date=('t_dat', 'max')
    ).sort_values(
        ['customer_key', 'purchase_count', 'last_date', 'article_num'],
        ascending=[True, False, False, True], kind='stable'
    )
    counts['_user_rank'] = counts.groupby('customer_key', sort=False).cumcount() + 1
    user_items = counts[counts['_user_rank'] <= 12].copy()
    user_items = user_items[user_items['customer_key'].isin(target_user_set)]
    current_users = set(user_items['customer_key']) - assigned_users
    user_items = user_items[user_items['customer_key'].isin(current_users)]
    popular = window['article_num'].value_counts().head(12).rename_axis('article_num').reset_index(name='popularity_count')
    popular['_user_rank'] = 13 + np.arange(len(popular), dtype=np.int16)
    fallback = pd.DataFrame({'customer_key': np.sort(np.fromiter(current_users, dtype=np.uint64, count=len(current_users)))}).merge(popular, how='cross')
    selected = pd.concat([
        user_items[['customer_key', 'article_num', '_user_rank']],
        fallback[['customer_key', 'article_num', '_user_rank']],
    ], ignore_index=True).drop_duplicates(['customer_key', 'article_num'], keep='first')
    selected = top_k_by_user(selected, '_user_rank', top_n=12, ascending=True)
    selected['score'] = 1000 - selected['_user_rank'].astype('float32')
    baseline1_chunks.append(selected[['customer_key', 'article_num', 'score']])
    assigned_users.update(current_users)
   
remaining_users = target_user_set - assigned_users
if remaining_users:
    latest_window = history[history['t_dat'] >= TARGET_START - pd.Timedelta(days=7)]
    popular = latest_window['article_num'].value_counts().head(12).rename_axis('article_num').reset_index(name='popularity_count')
    popular['_user_rank'] = 1 + np.arange(len(popular), dtype=np.int16)
    fallback = pd.DataFrame({'customer_key': np.sort(np.fromiter(remaining_users, dtype=np.uint64, count=len(remaining_users)))}).merge(popular, how='cross')
    fallback['score'] = 1000 - fallback['_user_rank'].astype('float32')
    baseline1_chunks.append(fallback[['customer_key', 'article_num', 'score']])
baseline1_candidates = pd.concat(baseline1_chunks, ignore_index=True)
   
# Baseline 2: recent purchases + strongest same-day companion + popularity fallback.
same_day_rules = pd.read_parquet(MINING_DIR / 'association_rules_same_day.parquet')
best_companion = same_day_rules.sort_values(
    ['antecedent_num', 'pair_count', 'interest_score', 'consequent_num'],
    ascending=[True, False, False, True], kind='stable'
).drop_duplicates('antecedent_num')[['antecedent_num', 'consequent_num', 'pair_count', 'confidence', 'lift']]
recent7 = history[history['t_dat'] >= TARGET_START - pd.Timedelta(days=7)]
recent_counts = recent7.groupby(['customer_key', 'article_num'], as_index=False, observed=True).agg(
    purchase_count=('t_dat', 'size'), last_date=('t_dat', 'max')
).sort_values(
    ['customer_key', 'purchase_count', 'last_date', 'article_num'],
    ascending=[True, False, False, True], kind='stable'
)
recent_counts['_source_rank'] = recent_counts.groupby('customer_key', sort=False).cumcount() + 1
recent_counts = recent_counts[recent_counts['customer_key'].isin(target_user_set) & (recent_counts['_source_rank'] <= 12)].copy()
companion = recent_counts.merge(best_companion, left_on='article_num', right_on='antecedent_num', how='inner')
companion = companion.rename(columns={'consequent_num': 'candidate_article'})
companion['_source_rank'] = 20 + companion.groupby('customer_key', sort=False).cumcount() + 1
baseline2_source = pd.concat([
    recent_counts[['customer_key', 'article_num', '_source_rank']],
    companion[['customer_key', 'candidate_article', '_source_rank']].rename(columns={'candidate_article': 'article_num'}),
], ignore_index=True).drop_duplicates(['customer_key', 'article_num'], keep='first')
baseline2_source = top_k_by_user(baseline2_source, '_source_rank', top_n=12, ascending=True)
covered_b2_users = set(baseline2_source['customer_key'])
popular7 = recent7['article_num'].value_counts().head(12).rename_axis('article_num').reset_index(name='popularity_count')
popular7['_source_rank'] = 40 + np.arange(len(popular7), dtype=np.int16)
b2_counts = baseline2_source.groupby("customer_key").size()
remaining_b2 = target_user_set - set(b2_counts[b2_counts >= 12].index)
if remaining_b2:
    b2_fallback = pd.DataFrame({'customer_key': np.sort(np.fromiter(remaining_b2, dtype=np.uint64, count=len(remaining_b2)))}).merge(popular7, how='cross')
    baseline2_source = pd.concat([baseline2_source, b2_fallback[['customer_key', 'article_num', '_source_rank']]], ignore_index=True)
baseline2_source = baseline2_source.drop_duplicates(['customer_key', 'article_num'], keep='first')
baseline2_source = top_k_by_user(baseline2_source, '_source_rank', top_n=12, ascending=True)
baseline2_candidates = baseline2_source.copy()
baseline2_candidates['score'] = 1000 - baseline2_candidates['_source_rank'].astype('float32')
baseline2_candidates = baseline2_candidates[['customer_key', 'article_num', 'score']]
   
baseline1_full = evaluate_candidates('baseline1_recency', baseline1_candidates, target_users)
baseline1_valid = evaluate_candidates('baseline1_recency_valid', baseline1_candidates, valid_users)
baseline2_full = evaluate_candidates('baseline2_co_purchase', baseline2_candidates, target_users)
baseline2_valid = evaluate_candidates('baseline2_co_purchase_valid', baseline2_candidates, valid_users)
baseline_results = pd.DataFrame([baseline1_full, baseline1_valid, baseline2_full, baseline2_valid])
baseline_results.to_csv(TABLE_DIR / 'baseline_offline_evaluation.csv', index=False)
baseline_summary = {
    'baseline1_full_map@12': baseline1_full['map@12'],
    'baseline1_valid_map@12': baseline1_valid['map@12'],
    'baseline2_full_map@12': baseline2_full['map@12'],
    'baseline2_valid_map@12': baseline2_valid['map@12'],
    'train_users': int(len(train_users)),
    'valid_users': int(len(valid_users)),
    'elapsed_seconds': round(time.time() - baseline_started, 3),
}
print(json.dumps(baseline_summary, ensure_ascii=False, indent=2))

{"channel": "baseline1_recency", "map@12": 0.025135722147123254, "recall@12": 0.05044457806294796, "hit_rate@12": 0.10112489852719471, "covered_users": 68984, "evaluated_users": 68984}
{"channel": "baseline1_recency_valid", "map@12": 0.027734496179554247, "recall@12": 0.05306096089509892, "hit_rate@12": 0.10263138712734164, "covered_users": 13719, "evaluated_users": 13719}
{"channel": "baseline2_co_purchase", "map@12": 0.022838636177194508, "recall@12": 0.04890996520730041, "hit_rate@12": 0.09854459004986664, "covered_users": 68984, "evaluated_users": 68984}
{"channel": "baseline2_co_purchase_valid", "map@12": 0.02491353313299502, "recall@12": 0.05071239318430279, "hit_rate@12": 0.09774764924557183, "covered_users": 13719, "evaluated_users": 13719}
{
  "baseline1_full_map@12": 0.025135722147123254,
  "baseline1_valid_map@12": 0.027734496179554247,
  "baseline2_full_map@12": 0.022838636177194508,
  "baseline2_valid_map@12": 0.02491353313299502,
  "train_users": 55265,
  "valid_users": 1

## 2. 五路召回候选池

最终召回通道为：时间衰减复购、同天关联、隐式协同过滤、年龄分群热门与全局热门。每名用户按融合优先级聚合后保留 Top100 候选。

In [3]:
candidate_started = time.time()
target_history = history[history["customer_key"].isin(target_user_set)].copy()
# Channel 1: time-decay repeat purchase, Top50.
repurchase_stats = target_history.groupby(['customer_key', 'article_num'], as_index=False, observed=True).agg(
    purchase_count=('t_dat', 'size'), last_date=('t_dat', 'max'), avg_price=('price', 'mean')
)
repurchase_stats['days_since_last'] = (TARGET_START - repurchase_stats['last_date']).dt.days.astype('int16')
repurchase_stats['repurchase_score'] = np.log1p(repurchase_stats['purchase_count']) * np.power(0.985, repurchase_stats['days_since_last']).astype('float32')
repurchase_candidates = top_k_by_user(repurchase_stats, 'repurchase_score', top_n=50)
   

# Channel 2: same-day association companions, Top20.
def association_candidates_from_rules(rules, channel_name, top_n=20):
    selected_rules = rules.sort_values(
        ['antecedent_num', 'interest_score', 'consequent_num'], ascending=[True, False, True], kind='stable'
    ).copy()
    selected_rules['rule_rank'] = selected_rules.groupby('antecedent_num', sort=False).cumcount() + 1
    selected_rules = selected_rules[selected_rules['rule_rank'] <= top_n]
    recent_items = history[
        (history['t_dat'] >= TARGET_START - pd.Timedelta(days=90)) & history['customer_key'].isin(target_user_set)
    ][['customer_key', 'article_num']].drop_duplicates()
    expanded = recent_items.merge(
        selected_rules[['antecedent_num', 'consequent_num', 'pair_count', 'confidence', 'lift', 'interest_score']],
        left_on='article_num', right_on='antecedent_num', how='inner'
    ).rename(columns={'consequent_num': 'candidate_article'})
    expanded = expanded.merge(
        history_pair_set.rename(columns={'article_num': 'candidate_article'}),
        on=['customer_key', 'candidate_article'], how='left'
    )
    expanded = expanded[expanded['in_history'].fillna(0) == 0]
    expanded[f'{channel_name}_score'] = np.log1p(expanded['pair_count']) * expanded['confidence'] * expanded['lift'].clip(upper=20)
    result = expanded.groupby(['customer_key', 'candidate_article'], as_index=False, observed=True).agg(
        score=(f'{channel_name}_score', 'sum'), rule_count=('interest_score', 'size'),
        max_confidence=('confidence', 'max'), max_lift=('lift', 'max'), max_pair_count=('pair_count', 'max')
    ).rename(columns={'candidate_article': 'article_num'})
    return top_k_by_user(result, 'score', top_n=top_n)
   
same_day_candidates = association_candidates_from_rules(same_day_rules, 'same_day_association', top_n=20)
   
# Channel 4: SVD item-item implicit CF, Top20.
item_cf_neighbors = pd.read_parquet(MINING_DIR / 'item_cf_neighbors.parquet')
cf_history = target_history[target_history['t_dat'] >= TARGET_START - pd.Timedelta(days=60)]
cf_interactions = cf_history.groupby(['customer_key', 'article_num'], as_index=False, observed=True).agg(
    purchase_count=('t_dat', 'size'), last_date=('t_dat', 'max')
)
cf_interactions['days_to_target'] = (TARGET_START - cf_interactions['last_date']).dt.days.astype('int16')
cf_interactions['recency_weight'] = np.power(0.995, cf_interactions['days_to_target']).astype('float32')
cf_expanded = cf_interactions.merge(
    item_cf_neighbors[['article_num', 'neighbor_num', 'similarity', 'similarity_rank']], on='article_num', how='inner'
).rename(columns={'neighbor_num': 'candidate_article'})
cf_expanded = cf_expanded.merge(history_pair_set.rename(columns={'article_num': 'candidate_article'}), on=['customer_key', 'candidate_article'], how='left')
cf_expanded = cf_expanded[cf_expanded['in_history'].fillna(0) == 0]
cf_expanded['cf_score'] = cf_expanded['similarity'] * cf_expanded['recency_weight'] / cf_expanded['similarity_rank']
cf_candidates = cf_expanded.groupby(['customer_key', 'candidate_article'], as_index=False, observed=True).agg(
    score=('cf_score', 'sum'), source_article_count=('article_num', 'nunique'), max_similarity=('similarity', 'max')
).rename(columns={'candidate_article': 'article_num'})
cf_candidates = top_k_by_user(cf_candidates, 'score', top_n=20)
   

# Channel 4: age-group segmented popularity.
customer_mining_profile = pd.read_parquet(MINING_DIR / 'customer_mining_profile.parquet')
segment_popularity = pd.read_parquet(MINING_DIR / 'segment_popularity_top50.parquet')
def segmented_candidates(segment_name, top_n):
    mapping = customer_mining_profile[customer_mining_profile['customer_key'].isin(target_user_set)][['customer_key', segment_name]].copy()
    mapping['segment_value'] = mapping[segment_name].astype(str)
    selected = segment_popularity[(segment_popularity['segment_name'] == segment_name) & (segment_popularity['popularity_rank'] <= top_n)]
    expanded = mapping[['customer_key', 'segment_value']].merge(
        selected[['segment_value', 'article_num', 'segment_purchase_count', 'popularity_rank']], on='segment_value', how='inner'
    )
    expanded = expanded.merge(history_pair_set, on=['customer_key', 'article_num'], how='left')
    expanded = expanded[expanded['in_history'].fillna(0) == 0]
    expanded[f'{segment_name}_score'] = expanded['segment_purchase_count'] / expanded['popularity_rank']
    result = expanded.groupby(['customer_key', 'article_num'], as_index=False, observed=True).agg(score=(f'{segment_name}_score', 'sum'))
    return top_k_by_user(result, 'score', top_n=top_n)
   
age_candidates = segmented_candidates('age_group', 30)
   
# Channel 5: global popularity fallback, Top30.
global_popular = history[history['t_dat'] >= TARGET_START - pd.Timedelta(days=7)]['article_num'].value_counts().head(30).rename_axis('article_num').reset_index(name='score')
global_candidates = target_user_frame.merge(global_popular, how='cross')
global_candidates = global_candidates.merge(history_pair_set, on=['customer_key', 'article_num'], how='left')
global_candidates = global_candidates[global_candidates['in_history'].fillna(0) == 0].drop(columns='in_history')
   
channel_specs = {
    'repurchase': (repurchase_candidates.rename(columns={'repurchase_score': 'score'}), 1.00),
    'same_day_association': (same_day_candidates, 0.70),
    'implicit_cf': (cf_candidates, 0.50),
    'age_popularity': (age_candidates, 0.45),
    'global_popularity': (global_candidates, 0.20),
}
source_frames = []
for channel, (frame, weight) in channel_specs.items():
    normalized = frame[['customer_key', 'article_num', 'score']].copy()
    normalized = normalized.sort_values(['customer_key', 'score', 'article_num'], ascending=[True, False, True], kind='stable')
    normalized['source_rank'] = normalized.groupby('customer_key', sort=False).cumcount() + 1
    normalized['source_score'] = normalized['score'].astype('float32')
    normalized['source_channel'] = channel
    normalized['source_priority'] = weight / normalized['source_rank'].astype('float32')
    source_frames.append(normalized)
all_sources = pd.concat(source_frames, ignore_index=True)
candidate_pool = all_sources.groupby(['customer_key', 'article_num'], as_index=False, observed=True).agg(
    source_count=('source_channel', 'nunique'), source_priority=('source_priority', 'sum'),
    best_source_rank=('source_rank', 'min')
)
candidate_pool = candidate_pool.rename(columns={'source_priority': 'pool_score'})
candidate_pool = top_k_by_user(candidate_pool, 'pool_score', top_n=100)
candidate_pool = candidate_pool.merge(target_truth[['customer_key', 'article_num', 'label']], on=['customer_key', 'article_num'], how='left')
candidate_pool['label'] = candidate_pool['label'].fillna(0).astype('int8')
candidate_pool.to_parquet(MODEL_DIR / 'candidate_pool_top100.parquet', index=False, compression='zstd')
all_sources.to_parquet(MODEL_DIR / 'candidate_sources.parquet', index=False, compression='zstd')
   
pool_counts = candidate_pool.groupby('customer_key', observed=True).size()
pool_recall_result, pool_ranked = evaluate_candidates(
    'candidate_pool_recall@100', candidate_pool.rename(columns={'pool_score': 'score'}), target_users, k=100, return_frame=True
)
candidate_summary = {
    'pool_rows': int(len(candidate_pool)),
    'pool_users': int(candidate_pool['customer_key'].nunique()),
    'candidate_count_mean': float(pool_counts.mean()),
    'candidate_count_median': float(pool_counts.median()),
    'candidate_count_p05': float(pool_counts.quantile(0.05)),
    'candidate_count_p95': float(pool_counts.quantile(0.95)),
    'positive_candidates': int(candidate_pool['label'].sum()),
    'target_pairs': int(len(target_truth)),
    'pool_recall@100': float(pool_recall_result['recall@100']),
    **pool_recall_result,
    'source_channel_rows': {channel: int(len(frame)) for channel, (frame, _) in channel_specs.items()},
    'elapsed_seconds': round(time.time() - candidate_started, 3),
}
del pool_recall_result, pool_ranked
print(json.dumps(candidate_summary, ensure_ascii=False, indent=2))

{
  "pool_rows": 6047766,
  "pool_users": 68984,
  "candidate_count_mean": 87.66911167807028,
  "candidate_count_median": 100.0,
  "candidate_count_p05": 48.0,
  "candidate_count_p95": 100.0,
  "positive_candidates": 21913,
  "target_pairs": 213728,
  "pool_recall@100": 0.12424358267917754,
  "channel": "candidate_pool_recall@100",
  "map@100": 0.023491331127934417,
  "recall@100": 0.12424358267917754,
  "hit_rate@100": 0.24601356836367855,
  "covered_users": 68984,
  "evaluated_users": 68984,
  "source_channel_rows": {
    "repurchase": 2040991,
    "same_day_association": 1009716,
    "implicit_cf": 817263,
    "age_popularity": 2043394,
    "global_popularity": 2050934
  },
  "elapsed_seconds": 151.82
}


## 3. 最终 31 特征工程

特征只使用测试开始日之前的数据，覆盖用户-商品交互、商品多时间窗热度、类目亲和、用户画像、价格匹配与召回融合信号。

In [4]:
article_meta = articles[[
    'article_num', 'product_type_name', 'department_name', 'colour_group_name', 'index_group_name'
]]
feature_started = time.time()
# User behavioral features.
user_features = target_history.groupby('customer_key', as_index=False, observed=True).agg(
    user_purchase_count=('t_dat', 'size'), user_active_days=('t_dat', 'nunique'),
    user_unique_articles=('article_num', 'nunique'), user_first_date=('t_dat', 'min'),
    user_last_date=('t_dat', 'max'), user_avg_price=('price', 'mean'),
    user_max_price=('price', 'max'), user_min_price=('price', 'min')
)
user_features['user_recency_days'] = (TARGET_START - user_features['user_last_date']).dt.days.astype('int16')
user_features['user_history_span_days'] = (user_features['user_last_date'] - user_features['first_date'.replace('first_date', 'user_first_date')]).dt.days.astype('int16')
user_features['user_avg_items_per_active_day'] = user_features['user_purchase_count'] / user_features['user_active_days'].clip(lower=1)
user_channel = target_history.groupby('customer_key')['sales_channel_id'].agg(
    channel_1_count=lambda values: int((values == 1).sum()), channel_2_count=lambda values: int((values == 2).sum())
).reset_index()
user_features = user_features.merge(user_channel, on='customer_key', how='left')
user_features['user_channel2_share'] = user_features['channel_2_count'] / (user_features['channel_1_count'] + user_features['channel_2_count']).clip(lower=1)
   
# Article popularity features under multiple windows.
article_base = history.groupby('article_num', as_index=False, observed=True).agg(
    article_first_date=('t_dat', 'min'), article_last_date=('t_dat', 'max'),
    article_avg_price=('price', 'mean'), article_channel2_count=('sales_channel_id', lambda values: int((values == 2).sum()))
)
article_base['article_age_days'] = (TARGET_START - article_base['article_first_date']).dt.days.astype('int32')
article_base['article_days_since_last'] = (TARGET_START - article_base['article_last_date']).dt.days.astype('int16')
for window_days in [7, 14, 28, 91, 364]:
    window = history[history['t_dat'] >= TARGET_START - pd.Timedelta(days=window_days)]
    popularity = window.groupby('article_num', as_index=False, observed=True).agg(
        purchase_count=('t_dat', 'size'), customer_count=('customer_key', 'nunique'), revenue=('price', 'sum')
    ).rename(columns={
        'purchase_count': f'article_purchase_{window_days}d',
        'customer_count': f'article_customer_{window_days}d',
        'revenue': f'article_revenue_{window_days}d',
    })
    article_base = article_base.merge(popularity, on='article_num', how='left')
for column in article_base.columns:
    if column.startswith('article_purchase_') or column.startswith('article_customer_') or column.startswith('article_revenue_'):
        article_base[column] = article_base[column].fillna(0).astype('float32')
article_base['article_popularity_trend_7_28'] = (
    article_base['article_purchase_7d'] / article_base['article_purchase_28d'].clip(lower=1)
)
   
# Candidate base with article metadata.
candidate_features = candidate_pool.merge(article_meta, on='article_num', how='left')
candidate_features = candidate_features.merge(article_base.drop(columns=['article_first_date', 'article_last_date']), on='article_num', how='left')
   
# User-article interaction features.
interaction_features = target_history.groupby(['customer_key', 'article_num'], as_index=False, observed=True).agg(
    user_article_count=('t_dat', 'size'), user_article_first_date=('t_dat', 'min'),
    user_article_last_date=('t_dat', 'max'), user_article_avg_price=('price', 'mean')
)
interaction_features['user_article_days_since_last'] = (TARGET_START - interaction_features['user_article_last_date']).dt.days.astype('int16')
interaction_features['user_article_span_days'] = (interaction_features['user_article_last_date'] - interaction_features['user_article_first_date']).dt.days.astype('int16')
interaction_features['user_article_time_decay_score'] = np.log1p(interaction_features['user_article_count']) * np.power(0.985, interaction_features['user_article_days_since_last']).astype('float32')
candidate_features = candidate_features.merge(
    interaction_features.drop(columns=['user_article_first_date', 'user_article_last_date']),
    on=['customer_key', 'article_num'], how='left'
)
   
# User-category affinity features.
def category_affinity(category_column, prefix):
    counts = target_history.merge(article_meta[['article_num', category_column]], on='article_num', how='left').groupby(
        ['customer_key', category_column], observed=True
    ).agg(category_purchase_count=('t_dat', 'size'), category_unique_articles=('article_num', 'nunique')).reset_index()
    counts = counts.rename(columns={
        category_column: f'candidate_{category_column}',
        'category_purchase_count': f'{prefix}_purchase_count',
        'category_unique_articles': f'{prefix}_unique_articles',
    })
    return counts
   
for category_column, prefix in [
    ('index_group_name', 'user_index_group'),
    ('product_type_name', 'user_product_type'),
    ('colour_group_name', 'user_colour_group'),
]:
    affinity_table = category_affinity(category_column, prefix)
    candidate_features = candidate_features.merge(
        affinity_table,
        left_on=['customer_key', category_column],
        right_on=['customer_key', f'candidate_{category_column}'], how='left'
    )
   
# Source channel flags, ranks, and raw scores.
for channel in channel_specs:
    source = all_sources[all_sources['source_channel'] == channel][['customer_key', 'article_num', 'source_rank', 'source_score']]
    source = source.rename(columns={'source_rank': f'{channel}_rank', 'source_score': f'{channel}_score'})
    candidate_features = candidate_features.merge(source, on=['customer_key', 'article_num'], how='left')
    candidate_features[f'{channel}_flag'] = candidate_features[f'{channel}_rank'].notna().astype('int8')
   
# User profile and mined segment features.
profile_features = customers[['customer_key', 'age', 'club_member_status', 'fashion_news_frequency']].merge(
    customer_mining_profile[['customer_key', 'activity_segment', 'recency_segment', 'lifetime_purchases', 'lifetime_active_days']],
    on='customer_key', how='left'
)
profile_features['age_group'] = pd.cut(
    profile_features['age'], bins=[15, 25, 35, 45, 55, 65, 100],
    labels=['16-25', '26-35', '36-45', '46-55', '56-65', '66+']
).cat.add_categories('unknown').fillna('unknown')
candidate_features = candidate_features.merge(profile_features, on='customer_key', how='left')
candidate_features = candidate_features.merge(
    pd.read_parquet(MINING_DIR / 'customer_price_preferences.parquet')[['customer_key', 'median_price', 'mean_price', 'price_band']],
    on='customer_key', how='left'
)
   
# Price matching features.
candidate_features['price_diff'] = candidate_features['article_avg_price'] - candidate_features['median_price']
candidate_features['price_ratio'] = candidate_features['article_avg_price'] / candidate_features['median_price'].clip(lower=1e-5)
candidate_features['price_preference_match'] = 1 / (1 + np.abs(np.log1p(candidate_features['article_avg_price'].fillna(0)) - np.log1p(candidate_features['median_price'].fillna(0))))
   
# Keep only the delivered feature set.
final_features = [
    'user_article_time_decay_score', 'department_name', 'article_customer_14d',
    'colour_group_name', 'recency_segment', 'product_type_name', 'pool_score',
    'user_article_days_since_last', 'article_purchase_28d', 'lifetime_purchases',
    'article_revenue_7d', 'mean_price', 'article_popularity_trend_7_28',
    'user_product_type_purchase_count', 'age', 'lifetime_active_days',
    'user_index_group_unique_articles', 'user_article_avg_price',
    'user_colour_group_purchase_count', 'price_preference_match', 'price_ratio',
    'median_price', 'article_revenue_28d', 'age_group', 'article_purchase_91d',
    'article_revenue_91d', 'article_purchase_364d', 'article_revenue_364d',
    'article_avg_price', 'implicit_cf_score', 'global_popularity_score',
]
categorical_features = [
    'department_name', 'colour_group_name', 'recency_segment', 'product_type_name', 'age_group'
]
numeric_features = [column for column in final_features if column not in categorical_features]
assert len(final_features) == 31
assert len(categorical_features) == 5
assert all(column in candidate_features.columns for column in final_features)
for column in categorical_features:
    candidate_features[column] = candidate_features[column].astype('category')
for column in numeric_features:
    candidate_features[column] = (
        candidate_features[column].fillna(-1).replace([np.inf, -np.inf], -1).astype('float32')
    )
ranking_dataset = candidate_features[['customer_key', 'article_num', 'label'] + final_features]
ranking_dataset.to_parquet(FINAL_DIR / 'ranking_dataset.parquet', index=False, compression='zstd')
pd.DataFrame({'feature': final_features}).to_csv(FINAL_DIR / 'final_features.csv', index=False)
feature_summary = {
    'rows': int(len(candidate_features)),
    'users': int(candidate_features['customer_key'].nunique()),
    'articles': int(candidate_features['article_num'].nunique()),
    'numeric_features': len(numeric_features),
    'categorical_features': len(categorical_features),
    'total_features': len(final_features),
    'positive_rows': int(candidate_features['label'].sum()),
    'positive_rate': float(candidate_features['label'].mean()),
    'elapsed_seconds': round(time.time() - feature_started, 3),
}
print(json.dumps(feature_summary, ensure_ascii=False, indent=2))


{
  "rows": 6047766,
  "users": 68984,
  "articles": 76419,
  "numeric_features": 26,
  "categorical_features": 5,
  "total_features": 31,
  "positive_rows": 21913,
  "positive_rate": 0.003623321404961766,
  "elapsed_seconds": 131.877
}


## 4. LightGBM LambdaRank 最终模型

按用户分组训练排序模型，评估指标与竞赛主指标保持一致：MAP@12，同时报告 Recall@12、HitRate@12 和候选级 AUC。

In [5]:

model_started = time.time()
model_data = ranking_dataset.merge(user_fold, on='customer_key', how='inner')
train_data = model_data[model_data['fold'] == 'train'].sort_values(
    ['customer_key', 'article_num'], kind='stable'
).reset_index(drop=True)
valid_data = model_data[model_data['fold'] == 'valid'].sort_values(
    ['customer_key', 'article_num'], kind='stable'
).reset_index(drop=True)
train_groups = train_data.groupby('customer_key', sort=False, observed=True).size().to_numpy(dtype=np.int32)
valid_groups = valid_data.groupby('customer_key', sort=False, observed=True).size().to_numpy(dtype=np.int32)
train_dataset = lgb.Dataset(
    train_data[final_features], label=train_data['label'].astype('int8'), group=train_groups,
    feature_name=final_features, categorical_feature=categorical_features, free_raw_data=False,
)
valid_dataset = lgb.Dataset(
    valid_data[final_features], label=valid_data['label'].astype('int8'), group=valid_groups,
    reference=train_dataset, feature_name=final_features, categorical_feature=categorical_features,
    free_raw_data=False,
)
params = {
    'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [12],
    'lambdarank_truncation_level': 12, 'label_gain': [0, 1],
    'learning_rate': 0.03, 'num_leaves': 63, 'min_data_in_leaf': 100,
    'feature_fraction': 0.85, 'bagging_fraction': 0.90, 'bagging_freq': 1,
    'lambda_l1': 0.5, 'lambda_l2': 5.0, 'num_threads': 4,
    'verbosity': -1, 'seed': 2026,
}
final_model = lgb.train(
    params, train_dataset, num_boost_round=500,
    valid_sets=[train_dataset, valid_dataset], valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(80, verbose=False)],
)
valid_data['score'] = final_model.predict(
    valid_data[final_features], num_iteration=final_model.best_iteration
)
final_metrics = evaluate_candidates(
    'lightgbm_lambdarank_final_valid',
    valid_data[['customer_key', 'article_num', 'score']], valid_users,
)
final_metrics.update({
    'valid_candidate_auc': float(roc_auc_score(valid_data['label'].astype(int), valid_data['score'])),
    'best_iteration': int(final_model.best_iteration),
    'elapsed_seconds': round(time.time() - model_started, 3),
})

feature_importance = pd.DataFrame({
    'feature': final_features,
    'gain': final_model.feature_importance(importance_type='gain'),
    'split': final_model.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False).reset_index(drop=True)
feature_importance.to_csv(TABLE_DIR / 'final_model_feature_importance.csv', index=False)
final_model.save_model(
    str(FINAL_DIR / 'lightgbm_lambdarank_final.txt'),
    num_iteration=final_model.best_iteration,
)
valid_data.sort_values(
    ['customer_key', 'score', 'article_num'], ascending=[True, False, True], kind='stable'
).groupby('customer_key', sort=False, observed=True).head(12).to_parquet(
    FINAL_DIR / 'final_model_valid_top12.parquet', index=False, compression='zstd'
)

comparison = pd.DataFrame([baseline1_valid, baseline2_valid, final_metrics])
comparison['map_gain_vs_baseline1'] = comparison['map@12'] / baseline1_valid['map@12'] - 1
comparison['map_gain_vs_baseline2'] = comparison['map@12'] / baseline2_valid['map@12'] - 1
comparison.to_csv(TABLE_DIR / 'final_model_vs_baselines.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
comparison_plot = comparison.copy()
comparison_plot['model'] = ['Baseline 1', 'Baseline 2', 'Final model']
sns.barplot(
    data=comparison_plot, x='map@12', y='model', hue='model',
    ax=axes[0], palette='Set2', legend=False,
)
axes[0].set_title('Validation MAP@12')
axes[0].set_xlabel('MAP@12')
axes[0].set_ylabel('')
top_importance = feature_importance.head(25).iloc[::-1]
sns.barplot(
    data=top_importance, x='gain', y='feature', hue='feature',
    ax=axes[1], palette='Set2', legend=False,
)
axes[1].set_title('Final model feature importance')
axes[1].set_xlabel('Gain')
axes[1].set_ylabel('')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'final_model_summary.png', bbox_inches='tight')
plt.close(fig)

final_summary = {
    'generated_at': time.strftime('%Y-%m-%d %H:%M:%S %z'),
    'status': 'final',
    'model': 'LightGBM LambdaRank',
    'recall_channels': list(channel_specs.keys()),
    'feature_count': len(final_features),
    'candidate_rows': int(len(candidate_pool)),
    'target_period': [str(TARGET_START.date()), str(TARGET_END.date())],
    'evaluation_protocol': '80/20 customer-key split; validation users are absent from training; the same validation users are used for early stopping and reported metrics',
    'split_users': {'train': int(len(train_users)), 'valid': int(len(valid_users))},
    'parameters': params,
    'objective': params['objective'],
    'baselines': comparison.iloc[:2].to_dict(orient='records'),
    'final_model': final_metrics,
    'candidate_pool': candidate_summary,
    'features': feature_summary,
    'comparison': comparison.to_dict(orient='records'),
    'outputs': {
        'candidate_pool': str(FINAL_DIR / 'candidate_pool_top100.parquet'),
        'candidate_sources': str(FINAL_DIR / 'candidate_sources.parquet'),
        'ranking_dataset': str(FINAL_DIR / 'ranking_dataset.parquet'),
        'features': str(FINAL_DIR / 'final_features.csv'),
        'model': str(FINAL_DIR / 'lightgbm_lambdarank_final.txt'),
        'evaluation': str(TABLE_DIR / 'final_model_vs_baselines.csv'),
        'figure': str(FIGURE_DIR / 'final_model_summary.png'),
    },
}
(FINAL_DIR / 'final_model_summary.json').write_text(
    json.dumps(final_summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(comparison.to_string(index=False))
print(json.dumps(final_summary, ensure_ascii=False, indent=2))


{"channel": "lightgbm_lambdarank_final_valid", "map@12": 0.038757945492993995, "recall@12": 0.07566375556759065, "hit_rate@12": 0.1449814126394052, "covered_users": 13719, "evaluated_users": 13719}
                        channel   map@12  recall@12  hit_rate@12  covered_users  evaluated_users  valid_candidate_auc  best_iteration  elapsed_seconds  map_gain_vs_baseline1  map_gain_vs_baseline2
        baseline1_recency_valid 0.027734   0.053061     0.102631          13719            13719                  NaN             NaN              NaN               0.000000               0.113230
    baseline2_co_purchase_valid 0.024914   0.050712     0.097748          13719            13719                  NaN             NaN              NaN              -0.101713               0.000000
lightgbm_lambdarank_final_valid 0.038758   0.075664     0.144981          13719            13719             0.822039           226.0          201.647               0.397463               0.555698
{
  "generated

## 交付结论

- 五路召回覆盖复购、搭配、协同过滤、年龄个性化与冷启动热门信号；
- 31 个特征完整保留用户-商品交互、多时间窗商品热度、类目亲和、画像与价格匹配信息；
- LightGBM LambdaRank 直接优化 Top12 排序，所有评估均使用时间切分与用户级切分，避免验证期信息泄漏。